In [ ]:
import ee
import folium

# Authenticate and initialize Earth Engine
# ee.Authenticate()
ee.Initialize(project='ndvi-from-sentinel-2-463808')


In [ ]:
# Define ROI (Bwindi Impenetrable National Park rectangle)
roi = ee.Geometry.Rectangle([29.6, -1.1, 29.75, -0.95])


In [ ]:
# Load Sentinel-2 Harmonized SR collection filtered by ROI and date
filter_img2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

# Function to calculate NDVI and add as band
def addNDVI(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Map NDVI function over collection
s2_collection_with_ndvi = filter_img2_collection.map(addNDVI)

# Median composite of NDVI band clipped to ROI
median_ndvi = s2_collection_with_ndvi.select('NDVI').median().clip(roi)


In [ ]:
visualization_parameters = {
    'min': 0,
    'max': 1,
    'palette': ['red', 'yellow', 'green']
}


In [ ]:
# Convert Earth Engine geometry coordinates to Python object
coords = roi.coordinates().getInfo()[0]  # Get the first ring of rectangle

# Get bounds from the coordinate list
west_lon = coords[0][0]
south_lat = coords[0][1]
east_lon = coords[2][0]
north_lat = coords[2][1]

# Compute center
center_lat = (south_lat + north_lat) / 2
center_lon = (west_lon + east_lon) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)


In [ ]:
ndvi_mapid = median_ndvi.getMapId(visualization_parameters)

folium.TileLayer(
    tiles=ndvi_mapid['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Median NDVI',
    opacity=0.7
).add_to(m)


In [ ]:
folium.LayerControl().add_to(m)
m
